# Exploratory Data Analysis (EDA) for Liquefaction Assessment

> Understanding geotechnical data for machine learning

**Exercise:** [![Open in Colab](https://img.shields.io/badge/Open%20in-Colab-F9AB00?style=flat-square&logo=googlecolab)](https://colab.research.google.com/github/kks32-courses/ai-geotech/blob/main/docs/00-mlp-dtree/00-eda-liquefaction-exercise.ipynb)
**Solution:** [![Open in Colab](https://img.shields.io/badge/Open%20in-Colab-F9AB00?style=flat-square&logo=googlecolab)](https://colab.research.google.com/github/kks32-courses/ai-geotech/blob/main/docs/00-mlp-dtree/00-eda-liquefaction.ipynb)

---
## Introduction

Exploratory Data Analysis (EDA) is a critical first step in any machine learning project. For geotechnical engineering applications, EDA helps us:

- Understand the physical characteristics of our data
- Identify data quality issues (missing values, outliers, errors)
- Discover relationships between geotechnical parameters
- Make informed decisions about preprocessing and feature engineering
- Validate that data aligns with our engineering understanding

### Liquefaction and Lateral Spreading

Soil liquefaction occurs in saturated loose sandy soils subjected to rapid loading conditions, such as earthquakes. The generation of excess pore water pressure can lead to a sudden reduction in soil strength and stiffness. In gently sloping ground, this may generate lateral displacements known as **lateral spreading**.

### Dataset

We'll analyze the lateral spreading dataset from:

> Durante, M. G., & Rathje, E. M. (2021). An exploration of the use of machine learning to predict lateral spreading. *Earthquake Spectra*, 37(4), 2288-2314.

**Classification criteria**: Sites with >0.3 m lateral displacement are classified as experiencing lateral spreading (Target=1).

**Key features**:
- **GWD (m)**: Ground Water Depth - distance from surface to water table
- **L (km)**: Distance to nearest free face (river, channel)
- **Slope (%)**: Ground slope angle
- **PGA (g)**: Peak Ground Acceleration from earthquake
- **Elevation**: Site elevation (will be removed as Durante & Rathje found it less significant)
- **Target**: Binary classification (0 = no lateral spreading, 1 = lateral spreading)

In [ ]:
!pip3 install pandas numpy matplotlib seaborn scipy scikit-learn --quiet

## Load Data and Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

In [ ]:
# Load data
# 'https://raw.githubusercontent.com/kks32-courses/ai-geotech/refs/heads/main/docs/00-mlp-dtree/RF_YN_Model3.csv'

In [ ]:
# Dataset info


## Data Overview

In [ ]:
# Basic statistics


### Geotechnical Interpretation of Summary Statistics

Let's interpret these statistics from a geotechnical perspective:

- **GWD (Ground Water Depth)**: 
  - Range: Check if values are physically plausible (typically 0-5m for liquefaction-prone sites)
  - Mean vs. Median: Indicates distribution symmetry
  
- **PGA (Peak Ground Acceleration)**:
  - Typical range: 0.1g - 1.0g for significant earthquakes
  - Higher PGA → More severe shaking → Higher liquefaction risk
  
- **Slope**:
  - Range: Should be non-negative, typically 0-30%
  - Steeper slopes → More lateral spreading potential
  
- **Distance (L)**:
  - Sites closer to free faces show more lateral movement

In [ ]:
# Class balance
fig, ax = plt.subplots(1, 2, figsize=(10, 4))




ax[0].set_title('Class Distribution')
ax[0].set_xticklabels(['No Spreading', 'Spreading'], rotation=0)
ax[0].set_ylabel('Count')

df['Target'].value_counts(normalize=True).plot(kind='pie', ax=ax[1], autopct='%1.1f%%', 
                                                  colors=['steelblue', 'coral'])
ax[1].set_ylabel('')
plt.tight_layout()
plt.show()

print(f"\nClass balance: {df['Target'].value_counts(normalize=True).values}")

**Note on Class Imbalance**: If one class significantly outnumbers the other, we may need to consider:
- Stratified sampling during train-test split
- Class weighting in models
- Appropriate evaluation metrics (precision, recall, F1-score, AUC-ROC)

## Missing data analysis

In [ ]:
# Missing values
print("Missing Values:")





# Physical plausibility
print("\nPhysical Plausibility Checks:")
print(f"  Negative GWD: {(df['GWD (m)'] < 0).sum()}")
print(f"  Negative Slope: {(df['Slope (%)'] < 0).sum()}")
print(f"  Negative Distance: {(df['L (km)'] < 0).sum()}")
print(f"  PGA > 1.0g: {(df['PGA (g)'] > 1.0).sum()}")

# Duplicates



print(f"\nDuplicate rows: {dup} ({100*dup/len(df):.1f}%)")

### Handling Missing Data - Geotechnical Considerations

**Types of Missingness**:
1. **MCAR (Missing Completely At Random)**: No pattern to missing data
2. **MAR (Missing At Random)**: Missingness related to observed variables
3. **MNAR (Missing Not At Random)**: Missingness related to the missing value itself

**Strategies for Geotechnical Data**:
- **Deletion**: Remove samples if <5% missing and MCAR
- **Imputation**: 
  - Mean/Median: For continuous variables (GWD, PGA)
  - Mode: For categorical variables (soil type)
  - Domain knowledge: Use typical values from similar geological conditions
  - Advanced: KNN or iterative imputation
- **Keep as feature**: Sometimes missingness itself is informative (e.g., no GWD measurement might indicate deep water table)

## Feature Distributions by Class

**Key question:** Can we visually distinguish spreading from non-spreading sites?

In [ ]:
# Histograms comparing spreading vs non-spreading
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for idx, feature in enumerate(features):
    # Plot both classes
    df[df['Target']==0][feature].hist(ax=axes[idx], bins=30, alpha=0.6, 
                                       label='No Spreading', color='steelblue', edgecolor='black')
    df[df['Target']==1][feature].hist(ax=axes[idx], bins=30, alpha=0.6, 
                                       label='Spreading', color='coral', edgecolor='black')
    
    axes[idx].set_xlabel(feature, fontweight='bold')
    axes[idx].set_ylabel('Frequency')
    axes[idx].legend()
    axes[idx].grid(alpha=0.3)

plt.suptitle('Feature Distributions by Lateral Spreading', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

### Understanding Skewness and Kurtosis

**Skewness** measures asymmetry of the distribution:
- Skewness ≈ 0: Symmetric distribution (normal)
- Skewness > 0: Right-skewed (tail extends to right) - e.g., extreme high PGA values
- Skewness < 0: Left-skewed (tail extends to left)

**Kurtosis** measures the "tailedness" of the distribution:
- Kurtosis ≈ 0: Normal distribution
- Kurtosis > 0: Heavy tails (more outliers) - leptokurtic
- Kurtosis < 0: Light tails (fewer outliers) - platykurtic

**Geotechnical Implications**:
- Skewed data might need transformation (log, sqrt) for some models
- High kurtosis indicates potential outliers or extreme events (major earthquakes)
- Tree-based models handle skewness well; neural networks may benefit from normalization

## Quartiles and Box Plots by Class

**Quartiles:** Robust statistics not affected by outliers

- Q1, Q2 (median), Q3 divide data into 4 equal parts  
- IQR = Q3 - Q1 (middle 50% of data)
- Outlier rule: Beyond Q1 - 1.5*IQR or Q3 + 1.5*IQR

In [ ]:
# Box plots by class
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

for idx, feature in enumerate(features):
    df.boxplot(column=feature, by='Target', ax=axes[idx], patch_artist=True)
    axes[idx].set_title(f'{feature}')
    axes[idx].set_xlabel('')
    axes[idx].set_xticklabels(['No Spread', 'Spread'])
    axes[idx].get_figure().suptitle('')

plt.suptitle('Box Plots by Target Class', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Compute quartiles
print("\nFive-Number Summary:\n")
print(f"{'Feature':<12} {'Min':>8} {'Q1':>8} {'Median':>8} {'Q3':>8} {'Max':>8} {'IQR':>8}")
print("-" * 70)





    
    print(f"{feature:<12} {df[feature].min():>8.2f} {q1:>8.2f} {median:>8.2f} {q3:>8.2f} {df[feature].max():>8.2f} {iqr:>8.2f}")

### What Do These Numbers Mean?

**Reading the Five-Number Summary:**

**GWD (Ground Water Depth):**
- **Middle 50%:** Between 1.64 m and 2.46 m (IQR = 0.82 m)
- **Median = 1.98 m:** Typical water table depth
- **Interpretation:** Most sites have water tables around 2 m deep, with relatively tight clustering (IQR = 0.82 m)

**L (Distance to Free Face):**
- **Middle 50%:** Between 0.49 km and 1.48 km (IQR = 0.99 km)  
- **Median = 0.95 km:** Typical distance to river/channel
- **Interpretation:** Most sites are roughly 0.5-1.5 km from water bodies

**Slope:**
- **Middle 50%:** Between 0.45% and 1.40% (IQR = 0.95%)
- **Median = 0.81%:** Very gentle slopes
- **Note:** Max = 10.92% shows some steep sites exist (but rare)

**PGA (Peak Ground Acceleration):**
- **Middle 50%:** Between 0.41g and 0.47g (IQR = 0.06g)
- **Median = 0.44g:** Moderate earthquake shaking
- **Interpretation:** Very tight clustering (IQR = 0.06g) - earthquakes in this dataset are similar strength

**Key Insight:** IQR tells you the SPREAD of the middle 50% of data
- **Small IQR** (like PGA = 0.06g): Data is tightly clustered → consistent conditions
- **Large IQR** (like L = 0.99 km): Data is more spread out → variable conditions

### Quick Reference - p-value interpretation

- p < 0.001 (***): Extremely strong evidence  
- p < 0.01 (**): Strong evidence  
- p < 0.05 (*): Moderate evidence
- p ≥ 0.05: No significant difference

h**The Rule:** Point is an outlier if:
- Below: Q1 - 1.5 × IQR  
- Above: Q3 + 1.5 × IQR

**Why 1.5?** John Tukey's empirical rule - balances detection vs false alarms

**What's a good outlier percentage?**
- 1-5%: Typical for clean data
- 5-10%: Acceptable, natural variability
- $>10\%$: Investigate - may indicate data quality issues or highly variable conditions

## Statistical Significance Testing

**The Question:** Are features truly different between spreading and non-spreading sites?

### Mann-Whitney U Test - How It Works

U counts the "wins" - how often one group beats the other.

**The Algorithm:**

1. **Mix all values** from both groups together
2. **Rank them** from smallest to largest (1, 2, 3, ...)
3. **Sum ranks** for each group
4. **Calculate U:** How often does a spreading value < non-spreading value?

**The p-value answers:** "If groups were actually the same, what's the chance we'd see this much separation by random luck?"
- p < 0.001 → Less than 0.1% chance → **Strong evidence of real difference!**
- p > 0.05 → More than 5% chance → **Could just be randomness**

**Why it's awesome:**
- ✓ No assumptions about distribution shape
- ✓ Robust to outliers (uses ranks, not values)
- ✓ Perfect for comparing two groups

In [ ]:
# Mann-Whitney U test (non-parametric)
from scipy.stats import mannwhitneyu

print("Mann-Whitney U Test Results:\n")
print(f"{'Feature':<12} {'p-value':>12} {'Significant?':>15}")
print("-" * 45)






    sig = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "No"
    
    print(f"{feature:<12} {p_value:>12.2e} {sig:>15}")

print("\n*** p<0.001  ** p<0.01  * p<0.05")

### Analysis of Results

**Key Insight:** Slope is NOT significantly different → May not help prediction much!

The other three features show extremely low p-values → These are our key predictors.

## Z-Score Method

**Z-Score:** Measures how many standard deviations from mean

$$z = \frac{x - \mu}{\sigma}$$

**Two uses:**
1. **Outlier detection:** |z| > 3 (rare values)
2. **Standardization for ML:** Transform to mean=0, std=1 (needed for neural networks, SVM)

**Rule:** |z| > 3 means outlier (only 0.3% of normal data exceeds this)

In [ ]:
# Simple IQR outlier detection
print("IQR Outlier Detection:\n")
print(f"{'Feature':<12} {'Q1':>8} {'Q3':>8} {'IQR':>8} {'Outliers':>10} {'%':>8}")
print("-" * 60)

for feature in features:





    
    print(f"{feature:<12} {q1:>8.2f} {q3:>8.2f} {iqr:>8.2f} {n_outliers:>10} {pct:>7.1f}%")

### Evaluating IQR Results

**Analysis:**

1. **GWD (1.8% outliers):** Valid extreme water table depths → **Keep**
2. **L (0.1% outliers):** Very few unusual distances → **Keep**
3. **Slope (7.4% outliers):** Natural field variability → **Keep**
4. **PGA (0.0% outliers):** Perfect - earthquake data well-controlled → **Keep**

**Decision:** All outliers < 10% and physically plausible → **Keep all data**

**Why?** Model needs to learn extreme events (major earthquakes, steep slopes)

### Method 2: Z-Score Method (Standardization)

The Z-score method identifies outliers as points with |z-score| > 3.

**What is a Z-score?**

The z-score measures how many standard deviations a data point is from the mean:

$$z = \frac{x - \mu}{\sigma}$$

where:
- $x$ = individual data point
- $\mu$ = population mean (estimated by sample mean $\bar{x}$)
- $\sigma$ = population standard deviation (estimated by sample std $s$)

**Interpretation:**
- $z = 0$: Value is exactly at the mean
- $z = 1$: Value is 1 standard deviation above the mean
- $z = -2$: Value is 2 standard deviations below the mean
- $|z| > 3$: Commonly used threshold for outlier detection (>99.7% of data falls within ±3σ for normal distribution)

**Why use Z-scores?**
1. **Dimensionless**: Allows comparison across different units (meters, g-forces, etc.)
2. **Standardization**: Transforms all features to same scale (mean=0, std=1)
3. **Statistical foundation**: Based on properties of normal distribution

**Limitations:**
- Assumes approximately normal distribution
- Sensitive to extreme outliers (they affect μ and σ)
- May not work well for highly skewed distributions

**Geotechnical Application:**
For example, if GWD has mean 2.0m and std 0.64m:
- A value of 3.28m would have z = (3.28-2.0)/0.64 = 2.0 (within normal range)
- A value of 6.0m would have z = (6.0-2.0)/0.64 = 6.25 (likely an outlier)

In [ ]:
# Z-score outliers
print("Z-Score Outlier Detection (|z| > 3):\n")
print(f"{'Feature':<12} {'Mean':>8} {'Std':>8} {'Outliers':>10} {'%':>8}")
print("-" * 50)

for feature in features:
    

    
    n_outliers = len(outliers)
    pct = 100 * n_outliers / len(df)
    
    print(f"{feature:<12} {df[feature].mean():>8.2f} {df[feature].std():>8.2f} {n_outliers:>10} {pct:>7.1f}%")

### Decision Framework for Outliers

**Keep outliers if:**
- Values are physically plausible (within engineering bounds)
- Represent real extreme events (major earthquakes, unusual site conditions)
- Only detected by one method (might be boundary cases)
- Critical for model to learn rare but important scenarios

**Remove outliers if:**
- Physically impossible values (negative GWD, PGA > 2g)
- Obvious data entry or measurement errors
- Detected by multiple methods consistently
- Would severely bias the model

**For this dataset:** Based on the geotechnical evaluation above, most outliers appear to be valid extreme events. We'll keep them for model training but flag them for monitoring.

## Correlation Analysis

In [ ]:
# Correlation matrix




plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nCorrelations with Target:")
print(corr['Target'].sort_values(ascending=False))

## Pair Plot - All Relationships at Once

In [ ]:
# Pair plot
g = sns.pairplot(df[features + ['Target']], hue='Target', 
                  palette=['steelblue', 'coral'],
                  plot_kws={'alpha': 0.5}, height=2)
g.fig.suptitle('Pairwise Feature Relationships', y=1.02, fontsize=14, fontweight='bold')
plt.show()

## Summary: Key Insights

**Most Important Features** (based on correlation and statistical tests):
1. **GWD**: Shallow water table → more spreading
2. **PGA**: Higher shaking → more spreading  
3. **L**: Closer to river → more spreading
4. **Slope**: Effect less clear

**Data Quality**: ✓ Good - no missing values, physically plausible

**Next Steps**: Ready for ML modeling!